# 第四章：符号域音乐表示与分析工具补充示例

本 Notebook 补充正文“符号域音乐分析”一节，展示 `music21` 之外的若干可调用研究工具和框架。

- **BACHI**：直接读取 MIDI / MusicXML，转换为 piano roll，并用预训练 Transformer 做符号域和弦识别。
- **Mel2Word**：其官方仓库提供 MIDI melody 到 Mel2Word representation 的代码；本 Notebook 只复刻基础编码与字典最长匹配，不能视为完整官方流水线。
- **MusicLang**：面向调性音乐的 Python language / framework，可加载、书写、变换、预测符号音乐，并创建可解释的 MIDI 文本表示。

> 注意：BACHI、Mel2Word、MusicLang 的定位并不相同。BACHI 是和弦识别模型；Mel2Word 是旋律表示方法；MusicLang 是调性音乐语言框架。

## 0. 环境与素材

输入素材为 `CODE/datasets/midi_author/C_major_piano.mid`。
本地文件名为 `C_major_piano.mid`，片段音级材料也与 C 大调参照相容；由于项目未附该文件的作品来源说明，这里把 C 大调作为本例分析参照，而不是来源学鉴定。

本 Notebook 的分析范围为开头 8 小节，并以整小节线和半小节线作为 BACHI 和弦边界的时间参照。

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import Audio, FileLink, display
from music21 import converter, key as m21key

# 仅过滤 music21 写 MIDI 时“指定通道不可用、已改用通道 1”的已知良性提示（TranslateWarning，UserWarning 子类），其余 UserWarning 保持可见。
warnings.filterwarnings(
    "ignore",
    message=r".* specified 1-indexed MIDI channel \d+ but acceptable channels were .*",
    category=UserWarning,
)


# 中文字体配置：优先使用 macOS 常见中文字体；若不可用，退回系统默认字体。
CHINESE_FONT_CANDIDATES = [
    "PingFang SC", "PingFang HK", "Heiti SC", "Heiti TC",
    "Hiragino Sans GB", "Songti SC", "STHeiti", "Arial Unicode MS",
    "Noto Sans CJK SC", "SimHei",
]
_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_chinese_font = next((name for name in CHINESE_FONT_CANDIDATES if name in _available_fonts), None)
if _chinese_font:
    plt.rcParams["font.sans-serif"] = [_chinese_font]
    plt.rcParams["font.family"] = "sans-serif"
    print("Matplotlib 中文字体:", _chinese_font)
else:
    print("未找到常见中文字体；若图中文字显示为方框，请安装 PingFang/Heiti/Noto Sans CJK。")
plt.rcParams["axes.unicode_minus"] = False

# 兼容从项目根目录或 CODE/chapter04 目录运行
cwd = Path.cwd().resolve()
if (cwd / "CODE" / "chapter04").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "chapter04" and cwd.parent.name == "CODE":
    PROJECT_ROOT = cwd.parents[1]
else:
    PROJECT_ROOT = cwd

CHAPTER_DIR = PROJECT_ROOT / "CODE" / "chapter04"
DATASET_DIR = PROJECT_ROOT / "CODE" / "datasets"
OUTPUT_FIG_DIR = CHAPTER_DIR / "output_figures"
OUTPUT_MIDI_DIR = CHAPTER_DIR / "output_midi"
OUTPUT_TEXT_DIR = CHAPTER_DIR / "output_text"
OUTPUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MIDI_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TEXT_DIR.mkdir(parents=True, exist_ok=True)

MIDI_PATH = DATASET_DIR / "midi_author" / "C_major_piano.mid"
EXCERPT_MEASURES = 8
EXCERPT_MIDI_PATH = OUTPUT_MIDI_DIR / "C_major_piano_opening_8_measures.mid"

print("项目根目录:", PROJECT_ROOT)
print("输入 MIDI:", MIDI_PATH)
print("输出片段:", EXCERPT_MIDI_PATH)
assert MIDI_PATH.exists(), f"找不到输入 MIDI: {MIDI_PATH}"

In [ ]:
score = converter.parse(str(MIDI_PATH))
score.makeMeasures(inPlace=True)

# 直接取开头 8 小节，并采用文件名所示的 C 大调作为分析参照；不把它写成来源学鉴定。
tonality = m21key.Key("C")
excerpt = score.measures(1, EXCERPT_MEASURES)
excerpt.write("midi", fp=str(EXCERPT_MIDI_PATH))

print(f"原始乐曲时值: {float(score.duration.quarterLength):.1f} quarterLength")
print(f"截取片段: 第 1–{EXCERPT_MEASURES} 小节，时值 {float(excerpt.duration.quarterLength):.1f} quarterLength")
print(f"调性设定: {tonality.tonic.name} {tonality.mode}")
print(f"已写出: {EXCERPT_MIDI_PATH}")

# 打印第一声部的小节边界，方便后面和 BACHI 输出对照。
first_part = excerpt.parts[0] if excerpt.parts else excerpt
for measure in first_part.getElementsByClass("Measure"):
    print(f"m.{measure.number:02d}: offset={float(measure.offset):5.1f}, duration={float(measure.duration.quarterLength):3.1f}")

## 0.1 播放截取片段

下面播放写出的 8 小节 MIDI 片段。若浏览器不能直接播放 MIDI，代码会使用 `pretty_midi` 将 MIDI 合成为音频预览。

In [ ]:
print("播放片段:", EXCERPT_MIDI_PATH)
display(FileLink(str(EXCERPT_MIDI_PATH), result_html_prefix="MIDI 文件："))

try:
    import pretty_midi
    playback_audio = pretty_midi.PrettyMIDI(str(EXCERPT_MIDI_PATH)).synthesize(fs=22050)
    peak = float(np.max(np.abs(playback_audio))) if playback_audio.size else 0.0
    if peak > 0:
        playback_audio = 0.9 * playback_audio / peak
    display(Audio(playback_audio, rate=22050))
except Exception as exc:
    print(f"音频预览合成失败: {exc}")
    print("可以点击上方 MIDI 文件链接，在本地播放器或 DAW 中播放。")


## 1. BACHI（符号深度学习和弦识别）

**BACHI**（ICASSP 2026）是一种**符号域**的 Transformer 和弦识别模型，与第五章直接从音频计算 chroma 后做模板匹配的示例输入域不同：

| 维度 | 第五章的 24 类大小三和弦模板示例 | BACHI |
|:---|:---|:---|
| **输入** | 音频信号 → 12 维 `chroma_cqt` | **MIDI / MusicXML** → piano roll |
| **表示** | 每帧的音级能量分布 | 按时间量化的 pitch × time 钢琴卷帘 |
| **判定** | 与 24 个 maj/min 二值模板计算余弦相似度并逐帧取最大值 | Transformer + Masked Iterative Decoding |
| **输出** | 逐帧 root + maj/min | root + quality + **bass（转位）** + 边界检测 |

BACHI 使用 **Masked Iterative Decoding**：按当前预测置信度逐轮填充被遮蔽的和弦表示；这不是一个固定的“先 root、再 quality、最后 bass”手写顺序。

本 Notebook 不经过“音频 → MIDI”的转写步骤，而是直接把现成 MIDI 片段输入 BACHI，以检查符号域模型输出的和弦边界与标签。

In [ ]:
bachi_available = False
bachi_times, bachi_labels = None, None

try:
    import torch
    import yaml
    from pathlib import Path as _Path

    # 源码位置：优先使用用户本地克隆；也可改成自己的 BACHI 仓库路径。
    BACHI_REPO_CANDIDATES = [
        PROJECT_ROOT / "CODE" / "external" / "BACHI_Chord_Recognition",
        _Path("/tmp/BACHI_Chord_Recognition"),
    ]
    bachi_repo = next((p for p in BACHI_REPO_CANDIDATES if (p / "inference.py").exists()), None)
    if bachi_repo is None:
        raise FileNotFoundError(
            "未找到 BACHI 源码。可将仓库克隆到 CODE/external/BACHI_Chord_Recognition "
            "或 /tmp/BACHI_Chord_Recognition。"
        )
    sys.path.insert(0, str(bachi_repo))

    from inference import extract_pianoroll, predict_piece  # noinspection PyUnresolvedReferences
    from dataset import load_vocabs  # noinspection PyUnresolvedReferences
    from models.variants import build_model  # noinspection PyUnresolvedReferences

    # 权重位置：优先查 chapter04；若尚未整理，兼容读取 chapter05 已下载的权重。
    CKPT_CANDIDATES = [
        CHAPTER_DIR / "models_checkpoints" / "bachi_pop" / "pop909_film_kdec",
        PROJECT_ROOT / "CODE" / "chapter05" / "models_checkpoints" / "bachi_pop" / "pop909_film_kdec",
    ]
    _bachi_ckpt = next((p for p in CKPT_CANDIDATES if (p / "best_model.pt").exists()), None)
    if _bachi_ckpt is None:
        raise FileNotFoundError("未找到 BACHI 权重 best_model.pt。")

    with open(_bachi_ckpt / "config.yaml", "r") as _f:
        _bachi_cfg = yaml.load(_f, Loader=yaml.FullLoader)

    _device = torch.device("cpu")
    _use_key = (
        bool(_bachi_cfg.get("use_key", False))
        or bool(_bachi_cfg["training"].get("use_key", False))
        or bool(_bachi_cfg["model"].get("use_key", False))
    )
    _vocabs = load_vocabs(str(_bachi_ckpt / "vocab.pkl"))
    _bachi_model = build_model(_bachi_cfg["experiment"], _bachi_cfg["model"], _vocabs, use_key=_use_key).to(_device)
    _bachi_model.load_state_dict(torch.load(_bachi_ckpt / "best_model.pt", map_location=_device, weights_only=True))
    _bachi_model.eval()
    bachi_available = True

    print("BACHI: 模型加载成功")
    print("源码:", bachi_repo)
    print("权重:", _bachi_ckpt)
except Exception as _e:
    print(f"BACHI: 加载失败 ({_e})")

In [ ]:
# BACHI 格式转换
_BACHI_QUAL_MAP = {
    "M": "maj", "m": "min", "D7": "7", "M7": "maj7", "m7": "min7",
    "+": "aug", "+7": "aug7", "o": "dim", "o7": "dim7", "/o7": "hdim7",
    "mM7": "minmaj7", "sus2": "sus2", "sus4": "sus4", "other": "other", "N": "N",
}


def parse_bachi_output(text: str):
    times, labels = [], []
    for line in text.strip().split("\n"):
        if not line.strip():
            continue
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        t = float(parts[0])
        r, q, b = parts[1].split("_")
        qual = _BACHI_QUAL_MAP.get(q, q)
        label = "N" if r == "N" or qual == "N" else f"{r}{qual}" + (f"/{b}" if b != r else "")
        times.append(t)
        labels.append(label)
    return times, labels


def run_bachi_on_midi(midi_path):
    if not bachi_available:
        return None, None
    pr = extract_pianoroll(_Path(midi_path), resolution=_bachi_cfg["model"]["beat_resolution"])
    if pr is None:
        print("BACHI: piano roll 提取失败")
        return None, None
    pianoroll_t = torch.from_numpy(pr.T).float()
    result_text = predict_piece(pianoroll_t, _bachi_model, _bachi_cfg, _vocabs, _device, _use_key)
    if result_text is None:
        return None, None
    return parse_bachi_output(result_text)

In [ ]:
if bachi_available:
    bachi_times, bachi_labels = run_bachi_on_midi(str(EXCERPT_MIDI_PATH))

    if bachi_labels:
        print(f"BACHI 识别结果: {len(set(bachi_labels))} 种和弦，共 {len(bachi_labels)} 个事件")
        print("前 20 个和弦:")
        for t, lab in zip(bachi_times[:20], bachi_labels[:20]):
            print(f"  {t:6.2f} 拍  {lab}")
    else:
        print("BACHI 未输出有效结果")
else:
    print("BACHI 不可用，跳过符号域和弦识别运行")

## 2. 可视化：和弦边界是否接近小节线

项目没有为这段 MIDI 提供人工和弦边界标注。下图只把整小节线和半小节线画成时间参照，用来观察 BACHI 输出的分段位置；靠近这些参照线本身不证明边界正确。

因此，观察 BACHI 输出时应把标签的音乐学可解释性和边界位置分开记录，并在有人工标注时再计算准确率或边界指标。

In [ ]:
def estimate_measure_boundaries(measure_count=8, beats_per_measure=4.0):
    # BACHI 输出的位置单位与 piano roll 的拍/quarterLength 对齐，不是音频秒数。
    # 因此这里直接画 4/4 小节线与半小节线，避免与 pretty_midi 秒数混用。
    measures = np.arange(0, measure_count + 1) * beats_per_measure
    halves = measures[:-1] + beats_per_measure / 2
    return measures, halves


if bachi_labels:
    measure_lines, half_measure_lines = estimate_measure_boundaries(EXCERPT_MEASURES)

    unique_bachi = sorted(set(bachi_labels))
    bachi_to_int = {label: idx for idx, label in enumerate(unique_bachi)}
    y = [bachi_to_int[label] for label in bachi_labels]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.step(bachi_times, y, where="post", color="0.1", lw=1.8, label="BACHI")

    for x in half_measure_lines:
        ax.axvline(x, color="0.85", lw=0.8, ls=":")
    for x in measure_lines:
        ax.axvline(x, color="0.65", lw=1.0, ls="--")

    ax.set_xlim(0, EXCERPT_MEASURES * 4.0)
    ax.set_yticks(range(len(unique_bachi)))
    ax.set_yticklabels(unique_bachi, fontsize=9)
    ax.set_xlabel("位置（拍 / quarterLength）")
    ax.set_ylabel("BACHI 和弦")
    ax.set_title("BACHI 符号深度学习和弦识别：C_major_piano 开头 8 小节", loc="left")
    ax.legend(frameon=True, edgecolor="lightgray")
    ax.set_ylim(-0.5, len(unique_bachi) - 0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_DIR / "bachi_42_c_major_opening.png", dpi=600, bbox_inches="tight")
    plt.show()
else:
    print("没有 BACHI 输出，跳过可视化。")

## 3. BACHI 结果如何解读

BACHI 与 `music21.chordify()` 采用不同的建模假设：

- `music21.chordify()` 先做纵向切片，再把同一时刻的音集合解释成和弦，规则透明但容易受经过音、延留音影响。
- BACHI 把和弦识别视为序列预测任务，从 piano roll 预测边界、根音、性质与低音，并输出分段标签；是否比某个基线更准确或更连续，仍需同一数据上的人工标注评测。
- BACHI 仍然受训练数据、和弦词表与输入 MIDI 质量限制。官方仓库说明模型只在钢琴数据上训练，输入 MIDI 音高范围为 21--108；超出这些条件的输入需要另行验证。

在本章的比较中，`music21` 是透明、可解释的规则基线；BACHI 代表可调用但需要手动配置的近期研究模型。

## 4. Mel2Word：MIDI melody 到文本式旋律表示

Mel2Word 官方 README 将其核心功能概括为：**Convert MIDI melodies into Mel2Word representations**。

它是一种表示方式：把单声部旋律中的音高运动与节奏间隔编码为类似文本的符号序列；官方 README 明确展示字典、BPE 式 tokenization、WordCloud 与 Word2Vec 等用法。

Mel2Word 不是独立的音乐分析模型；它提供“旋律如何被转换为可分析的文本式表示”这一中间层。官方仓库：https://github.com/saebyulpark/Mel2word。该仓库当前根目录未提供许可证文件，GitHub API 也未识别许可证，因此不能仅凭公开可访问性推断其代码可按某个开源许可证复用。

> 下方实现依据仓库示例复刻基础编码，不调用官方完整预处理脚本。本例代码会把舍入后的音程限制到 -12…12 半音、把 IOI 限制到 0…4 个四分音符时值，并在输出中报告是否实际发生裁剪；这些边界不能反推为官方完整流程的唯一规范。

In [ ]:
from collections import Counter
from music21 import chord as m21chord


def extract_topline_melody(m21_stream, part_index=0):
    """从指定声部提取单声部旋律；若遇到和弦，取最高音。"""
    part = m21_stream.parts[part_index] if m21_stream.parts else m21_stream
    events = []
    for event in part.flatten().notes:
        if isinstance(event, m21chord.Chord):
            pitch = max(event.pitches, key=lambda p: p.midi)
        elif event.isNote:
            pitch = event.pitch
        else:
            continue
        events.append({
            "offset": float(event.offset),
            "duration": float(event.duration.quarterLength),
            "pitch_midi": int(pitch.midi),
            "pitch_name": pitch.nameWithOctave,
        })
    events.sort(key=lambda item: (item["offset"], item["pitch_midi"]))
    return events


def encode_mel2word_interval(delta):
    """本例编码：舍入后裁剪到 [-12, 12] 半音。"""
    delta = int(np.clip(round(delta), -12, 12))
    if delta > 0:
        return f"U{delta:02d}"
    if delta < 0:
        return f"D{-delta:02d}"
    return "E00"


def encode_mel2word_ioi(delta, note_quantize=0.25):
    """本例编码：量化后裁剪到 [0, 4] 个四分音符时值。"""
    quantized = note_quantize * round(float(delta) / note_quantize)
    quantized = float(np.clip(quantized, 0, 4))
    if note_quantize == 0.25:
        return f"{int(quantized * 100):03d}"
    if note_quantize == 0.125:
        return f"{int(quantized * 1000):04d}"
    return f"{int(quantized * 10000):05d}"


def midi_melody_to_mel2word_units(melody_events, note_quantize=0.25):
    units = []
    for previous, current in zip(melody_events[:-1], melody_events[1:]):
        pitch_token = encode_mel2word_interval(current["pitch_midi"] - previous["pitch_midi"])
        rhythm_token = encode_mel2word_ioi(current["offset"] - previous["offset"], note_quantize)
        units.append(pitch_token + rhythm_token)
    return units


melody_events = extract_topline_melody(excerpt, part_index=0)
raw_intervals = [cur["pitch_midi"] - prev["pitch_midi"] for prev, cur in zip(melody_events[:-1], melody_events[1:])]
raw_iois = [cur["offset"] - prev["offset"] for prev, cur in zip(melody_events[:-1], melody_events[1:])]
interval_clip_count = sum(1 for value in raw_intervals if round(value) < -12 or round(value) > 12)
ioi_clip_count = sum(1 for value in raw_iois if 0.25 * round(value / 0.25) < 0 or 0.25 * round(value / 0.25) > 4)
mel2word_units = midi_melody_to_mel2word_units(melody_events)
mel2word_counts = Counter(mel2word_units)

print(f"提取旋律事件: {len(melody_events)} 个")
print("前 16 个旋律音:", " ".join(event["pitch_name"] for event in melody_events[:16]))
print(f"Mel2Word 基础单元: {len(mel2word_units)} 个，类型数 {len(mel2word_counts)}")
print(f"裁剪计数: 音程={interval_clip_count}，IOI={ioi_clip_count}")
print("前 24 个 Mel2Word 单元:")
print(" ".join(mel2word_units[:24]))

mel2word_text_path = OUTPUT_TEXT_DIR / "C_major_piano_opening_mel2word.txt"
mel2word_text_path.write_text(" ".join(mel2word_units), encoding="utf-8")
print("已写出:", mel2word_text_path)


### 4.1 可选：使用 Mel2Word 仓库随附字典做最长优先匹配

Mel2Word 仓库随附预生成 dictionary；本单元用最长优先匹配复刻从基础 token 到 word-like token 的合并步骤。它不是对官方完整流程的逐行调用，也不等同于重新训练 BPE。

若已将官方仓库克隆到 `CODE/external/Mel2word` 或 `/tmp/Mel2word`，下面的单元会读取 `Dictionary/Dictionary_all.pkl`，对上一格得到的基础单元执行最长优先匹配。

In [ ]:
import pickle

mel2word_tokens = None
mel2word_tokens_path = None
MEL2WORD_REPO_CANDIDATES = [
    PROJECT_ROOT / "CODE" / "external" / "Mel2word",
    Path("/tmp/Mel2word"),
]
mel2word_repo = next((path for path in MEL2WORD_REPO_CANDIDATES if (path / "Dictionary" / "Dictionary_all.pkl").exists()), None)


def build_mel2word_dictionary_by_length(dictionary, dic_size=100, min_freq=10, max_length=10):
    selected = []
    for word, count in sorted(dictionary.items(), key=lambda item: item[1], reverse=True):
        length = len(word.split("_"))
        if 1 < length <= max_length and count > min_freq:
            selected.append((word, length))
            if len(selected) == dic_size:
                break
    by_length = {}
    for word, length in selected:
        by_length.setdefault(length, set()).add(word)
    return by_length


def tokenize_mel2word_units(units, dictionary_by_length):
    tokens = []
    index = 0
    lengths = sorted(dictionary_by_length, reverse=True)
    while index < len(units):
        matched = None
        for length in lengths:
            if index + length <= len(units):
                candidate = "_".join(units[index:index + length])
                if candidate in dictionary_by_length[length]:
                    matched = candidate
                    index += length
                    break
        if matched is None:
            matched = units[index]
            index += 1
        tokens.append(matched)
    return tokens


if mel2word_repo is None:
    print("未找到 Mel2Word 官方字典，跳过 BPE 式 tokenization。")
else:
    dictionary_path = mel2word_repo / "Dictionary" / "Dictionary_all.pkl"
    with open(dictionary_path, "rb") as handle:
        mel2word_dictionary = pickle.load(handle)
    dictionary_by_length = build_mel2word_dictionary_by_length(mel2word_dictionary, dic_size=100)
    mel2word_tokens = tokenize_mel2word_units(mel2word_units, dictionary_by_length)
    mel2word_tokens_path = OUTPUT_TEXT_DIR / "C_major_piano_opening_mel2word_tokens.txt"
    mel2word_tokens_path.write_text(" ".join(mel2word_tokens), encoding="utf-8")
    print("Mel2Word 官方字典:", dictionary_path)
    print(f"合并后 token 数: {len(mel2word_tokens)}，类型数 {len(set(mel2word_tokens))}")
    print("前 20 个 token:")
    print(" ".join(mel2word_tokens[:20]))
    print("已写出:", mel2word_tokens_path)


In [ ]:
plot_mel2word_sequence = mel2word_tokens if mel2word_tokens else mel2word_units
plot_mel2word_counts = Counter(plot_mel2word_sequence)
plot_label = "Mel2Word token" if mel2word_tokens else "Mel2Word 基础单元"

if plot_mel2word_counts:
    common_units = plot_mel2word_counts.most_common(12)
    labels = [item[0] for item in common_units][::-1]
    values = [item[1] for item in common_units][::-1]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(labels, values, color="#4C78A8")
    ax.set_xlabel("出现次数")
    ax.set_ylabel(plot_label)
    ax.set_title("C_major_piano 开头 8 小节：Mel2Word 表示频次", loc="left")
    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_DIR / "mel2word_token_counts.png", dpi=600, bbox_inches="tight")
    plt.show()
else:
    print("没有可统计的 Mel2Word 单元。")


## 5. MusicLang：调性音乐的高层语言框架

MusicLang 官方 README 将其定义为一个 Python framework，核心是实现一种面向 tonal music 的新语言。

它支持 load、write、transform 和 predict symbolic music；基础包 `musiclang` 主要提供语言、记谱、变换和 MIDI / MusicXML 相关能力，AI 预测能力已拆分到 `musiclang_predict`。

基础包可从 MIDI 读取片段，并导出可解释、信息较丰富的 MusicLang 文本表示。官方仓库：https://github.com/MusicLang/musiclang。仓库的 `LICENSE.md` 正文是 BSD 2-Clause；README 中的 BSD 3-Clause 说明与许可证原文冲突，本章按许可证文件口径记录，并保留这一元数据冲突。

In [ ]:
musiclang_score = None
musiclang_text_path = OUTPUT_TEXT_DIR / "C_major_piano_opening_musiclang.txt"

def report_musiclang_import_error(exc):
    message = str(exc)
    print(f"MusicLang 导入失败: {type(exc).__name__}: {message}")
    if "numpy.dtype size changed" in message or "binary incompatibility" in message:
        print("诊断: 当前 Jupyter 内核中某个二进制依赖与 NumPy 版本不匹配；这不是 MIDI 文件解析错误。")
        print("建议: 在当前内核对应的 Python 环境中重装 NumPy 与编译型依赖，或新建独立环境。")
        print("当前 Python:", sys.executable)
        print("不建议在本书主环境中强制重装 NumPy / pandas / scipy / scikit-learn；这可能影响 torch / BACHI。")
        print("推荐新建独立环境运行 MusicLang: conda create -n musiclang-demo python=3.11")
        print("进入该环境后安装: python -m pip install \"numpy==1.26.4\" \"pandas==1.5.3\" musiclang")
    elif isinstance(exc, ModuleNotFoundError):
        print("可安装基础包: pip install musiclang")
    else:
        print("可检查 musiclang 及其依赖是否与当前 Python / NumPy 版本匹配。")


try:
    from musiclang import Score
except Exception as exc:
    report_musiclang_import_error(exc)
else:
    try:
        musiclang_score = Score.from_midi(str(EXCERPT_MIDI_PATH), chord_range=(0, EXCERPT_MEASURES))
        print("MusicLang: MIDI 读取成功")
        print("前 4 个 MusicLang chord / bar 表示:")
        print(musiclang_score[:4])
        musiclang_score.to_text_file(str(musiclang_text_path), create_dir=True)
        print("已写出:", musiclang_text_path)
    except Exception as exc:
        print(f"MusicLang MIDI 解析失败: {type(exc).__name__}: {exc}")


## 6. 方法定位小结

| 工具 / 方法 | 官方定位 | 本 Notebook 中的作用 |
|:---|:---|:---|
| BACHI | 符号域和弦识别模型 | 从 MIDI 片段预测和弦标签与边界 |
| Mel2Word | Convert MIDI melodies into Mel2Word representations | 复刻基础编码和字典最长匹配；不是完整官方流程 |
| MusicLang | Python framework / language for tonal music | 可选单元：安装成功时从 MIDI 创建高层音乐语言表示；本次环境未安装 |

这三个例子覆盖了“模型识别”“新表示”“高层语言框架”三种路线。它们都处理符号音乐，但研究目标和工程成熟度不同。

In [ ]:
output_status = [
    (EXCERPT_MIDI_PATH, True),
    (OUTPUT_FIG_DIR / "bachi_42_c_major_opening.png", bool(bachi_labels)),
    (OUTPUT_FIG_DIR / "mel2word_token_counts.png", bool(plot_mel2word_counts)),
    (OUTPUT_TEXT_DIR / "C_major_piano_opening_mel2word.txt", True),
    (OUTPUT_TEXT_DIR / "C_major_piano_opening_mel2word_tokens.txt", mel2word_tokens_path is not None),
    (musiclang_text_path, musiclang_score is not None),
]

print("本次运行的输出状态：")
for path, produced_this_run in output_status:
    relative_path = path.relative_to(CHAPTER_DIR)
    if produced_this_run and path.exists():
        print(f"  {relative_path}  已生成 ({path.stat().st_size / 1024:.1f} KB)")
    elif produced_this_run:
        print(f"  {relative_path}  预期生成但文件缺失")
    else:
        print(f"  {relative_path}  本次未生成")
